
# Day 2 — PyTorch Autograd

## Goal
By the end of this notebook, you should understand:

- What a **gradient** is
- What `requires_grad=True` does
- What a **computational graph** is
- How `.backward()` works
- How to inspect gradients using `.grad`
- Why gradients accumulate
- How to reset gradients
- What `torch.no_grad()` and `.detach()` do
- How Autograd connects to neural-network training

The notebook is designed so you can run the cells **from top to bottom**.


In [1]:

import torch

print("PyTorch version:", torch.__version__)


PyTorch version: 2.11.0+cpu



## 1. What is a gradient?

Suppose:

\[
y = x^2
\]

The derivative is:

\[
\frac{dy}{dx} = 2x
\]

At:

\[
x = 3
\]

we get:

\[
\frac{dy}{dx} = 2(3) = 6
\]

A gradient tells us:

> If the input changes slightly, how much does the output change?

This is important in neural networks because training asks:

> If a weight changes slightly, how will the loss change?



## 2. Your first PyTorch gradient

`requires_grad=True` tells PyTorch to track operations involving the tensor so that derivatives can later be calculated.


In [2]:

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

print("x:", x)
print("y:", y)


x: tensor(3., requires_grad=True)
y: tensor(9., grad_fn=<PowBackward0>)



Notice that `y` has a `grad_fn`.

That means PyTorch remembers which operation created `y`.

Now we calculate the gradient.


In [3]:

y.backward()

print("Gradient dy/dx:", x.grad)


Gradient dy/dx: tensor(6.)



Expected result:

\[
\frac{dy}{dx} = 2x = 2(3) = 6
\]

So PyTorch should show approximately:

```text
tensor(6.)
```

### Important

- `y.backward()` **calculates** the gradients.
- `x.grad` **stores** the gradient of `y` with respect to `x`.



## 3. Computational Graph

Consider:

\[
a = x^2
\]

\[
b = 3a
\]

\[
y = b + 5
\]

Therefore:

\[
y = 3x^2 + 5
\]

PyTorch internally remembers the sequence of operations:

```text
x
│
│ square
▼
a = x²
│
│ × 3
▼
b = 3x²
│
│ + 5
▼
y = 3x² + 5
```

This record is the **computational graph**.


In [4]:

x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = a * 3
y = b + 5

print("a =", a)
print("b =", b)
print("y =", y)

print("\na.grad_fn =", a.grad_fn)
print("b.grad_fn =", b.grad_fn)
print("y.grad_fn =", y.grad_fn)


a = tensor(4., grad_fn=<PowBackward0>)
b = tensor(12., grad_fn=<MulBackward0>)
y = tensor(17., grad_fn=<AddBackward0>)

a.grad_fn = <PowBackward0 object at 0x7891ecd146d0>
b.grad_fn = <MulBackward0 object at 0x7891ecd146d0>
y.grad_fn = <AddBackward0 object at 0x7891ecd146d0>



### Manual derivative

\[
y = 3x^2 + 5
\]

Therefore:

\[
\frac{dy}{dx} = 6x
\]

At:

\[
x = 2
\]

\[
\frac{dy}{dx} = 12
\]


In [5]:

y.backward()

print("dy/dx =", x.grad)


dy/dx = tensor(12.)



## 4. A More Interesting Function

Consider:

\[
y = x^3 + 2x^2 + 5x
\]

Its derivative is:

\[
\frac{dy}{dx} = 3x^2 + 4x + 5
\]

At:

\[
x = 2
\]

\[
3(2^2) + 4(2) + 5 = 12 + 8 + 5 = 25
\]


In [6]:

x = torch.tensor(2.0, requires_grad=True)

y = x**3 + 2*x**2 + 5*x

y.backward()

print("y =", y)
print("dy/dx =", x.grad)


y = tensor(26., grad_fn=<AddBackward0>)
dy/dx = tensor(25.)



## 5. Multiple Variables

Suppose:

\[
z = x^2 + 3y
\]

We want:

\[
\frac{\partial z}{\partial x}
\]

and:

\[
\frac{\partial z}{\partial y}
\]

Manually:

\[
\frac{\partial z}{\partial x} = 2x
\]

\[
\frac{\partial z}{\partial y} = 3
\]

At \(x=2\):

\[
\frac{\partial z}{\partial x}=4
\]


In [7]:

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

z = x**2 + 3*y

z.backward()

print("dz/dx =", x.grad)
print("dz/dy =", y.grad)


dz/dx = tensor(4.)
dz/dy = tensor(3.)



## 6. Gradients Accumulate

A very important PyTorch behavior:

> Gradients are **added** to existing gradients.

They are not automatically replaced.

Let's see it.


In [8]:

x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print("After first backward():", x.grad)

# Build a new graph using the same x
y = x ** 2
y.backward()

print("After second backward():", x.grad)


After first backward(): tensor(4.)
After second backward(): tensor(8.)



For \(y=x^2\) at \(x=2\):

\[
\frac{dy}{dx}=4
\]

First backward pass:

```text
gradient = 4
```

Second backward pass adds another `4`:

```text
gradient = 8
```

This is called **gradient accumulation**.



## 7. Resetting Gradients

For an individual tensor, you can reset its gradient using:

```python
x.grad.zero_()
```

Later, when training neural networks, you will normally use:

```python
optimizer.zero_grad()
```


In [9]:

x.grad.zero_()

print("Gradient after zeroing:", x.grad)


Gradient after zeroing: tensor(0.)



## 8. Why Do We Usually Use Floating-Point Tensors?

Gradient-based optimization works with continuous quantities.

Therefore this is the standard form:

```python
torch.tensor(3.0, requires_grad=True)
```

rather than an integer tensor.


In [10]:

x = torch.tensor(3.0, requires_grad=True)

print(x)
print("dtype:", x.dtype)


tensor(3., requires_grad=True)
dtype: torch.float32



## 9. Turning Gradient Tracking Off

When a trained model is only being used for prediction, we usually do not need gradients.

We can use:

```python
with torch.no_grad():
```

This saves memory and computation.


In [11]:

x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print("y =", y)
print("y.requires_grad =", y.requires_grad)


y = tensor(9.)
y.requires_grad = False



## 10. `.detach()`

`.detach()` creates a tensor containing the same numerical value but disconnected from the current computational graph.


In [12]:

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
z = y.detach()

print("y =", y)
print("y.requires_grad =", y.requires_grad)

print("\nz =", z)
print("z.requires_grad =", z.requires_grad)


y = tensor(9., grad_fn=<PowBackward0>)
y.requires_grad = True

z = tensor(9.)
z.requires_grad = False



## 11. How Autograd Connects to Neural Networks

Consider a tiny model:

\[
\text{prediction} = wx
\]

where:

- \(w\) = model weight
- \(x\) = input

Suppose:

\[
x = 2
\]

\[
w = 3
\]

Then:

\[
prediction = 3 \times 2 = 6
\]

Suppose the correct target is:

\[
10
\]

We can measure the error using squared loss:

\[
Loss = (prediction - target)^2
\]

So:

\[
Loss = (6-10)^2 = 16
\]

Autograd can calculate how the loss changes with respect to \(w\).


In [13]:

x = torch.tensor(2.0)
target = torch.tensor(10.0)

w = torch.tensor(3.0, requires_grad=True)

prediction = w * x
loss = (prediction - target) ** 2

loss.backward()

print("Input x:", x)
print("Weight w:", w)
print("Prediction:", prediction)
print("Target:", target)
print("Loss:", loss)
print("Gradient dLoss/dw:", w.grad)


Input x: tensor(2.)
Weight w: tensor(3., requires_grad=True)
Prediction: tensor(6., grad_fn=<MulBackward0>)
Target: tensor(10.)
Loss: tensor(16., grad_fn=<PowBackward0>)
Gradient dLoss/dw: tensor(-16.)



### Manual calculation of the gradient

\[
L=(wx-y)^2
\]

Using the chain rule:

\[
\frac{dL}{dw} = 2(wx-y)x
\]

Substitute:

\[
w=3,\quad x=2,\quad y=10
\]

\[
\frac{dL}{dw} = 2(6-10)(2)
\]

\[
=2(-4)(2)
\]

\[
=-16
\]

So PyTorch should return:

```text
tensor(-16.)
```

The negative gradient means that increasing the weight \(w\) would reduce the loss locally.



# 12. Mini Gradient Descent Demo

Now we will actually use the gradient to update the weight.

The update rule is:

\[
w_{new}=w_{old}-\eta \frac{dL}{dw}
\]

where:

\[
\eta
\]

is the **learning rate**.


In [14]:

x = torch.tensor(2.0)
target = torch.tensor(10.0)

w = torch.tensor(3.0, requires_grad=True)

learning_rate = 0.1

prediction = w * x
loss = (prediction - target) ** 2

loss.backward()

print("Before update")
print("Weight:", w.item())
print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("Gradient:", w.grad.item())

# Update the weight without tracking this update in the computational graph
with torch.no_grad():
    w -= learning_rate * w.grad

print("\nAfter update")
print("Updated weight:", w.item())
print("New prediction:", (w * x).item())


Before update
Weight: 3.0
Prediction: 6.0
Loss: 16.0
Gradient: -16.0

After update
Updated weight: 4.599999904632568
New prediction: 9.199999809265137



The weight moved in the direction that should reduce the error.

This basic pattern:

```text
forward calculation
       ↓
calculate loss
       ↓
backward()
       ↓
inspect gradients
       ↓
update parameters
       ↓
reset gradients
```

is the foundation of neural-network training.



# 13. A Small Training Loop

Let's repeat the process automatically.

Our model is still:

\[
prediction=wx
\]

We want the model to learn a weight such that:

\[
2w \approx 10
\]

So the ideal value of \(w\) is:

\[
w=5
\]


In [15]:

x = torch.tensor(2.0)
target = torch.tensor(10.0)

w = torch.tensor(1.0, requires_grad=True)

learning_rate = 0.1

for epoch in range(10):
    prediction = w * x
    loss = (prediction - target) ** 2

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad

    # Reset accumulated gradient
    w.grad.zero_()

    print(
        f"Epoch {epoch + 1:02d} | "
        f"weight = {w.item():.4f} | "
        f"prediction = {(w * x).item():.4f} | "
        f"loss = {loss.item():.4f}"
    )


Epoch 01 | weight = 4.2000 | prediction = 8.4000 | loss = 64.0000
Epoch 02 | weight = 4.8400 | prediction = 9.6800 | loss = 2.5600
Epoch 03 | weight = 4.9680 | prediction = 9.9360 | loss = 0.1024
Epoch 04 | weight = 4.9936 | prediction = 9.9872 | loss = 0.0041
Epoch 05 | weight = 4.9987 | prediction = 9.9974 | loss = 0.0002
Epoch 06 | weight = 4.9997 | prediction = 9.9995 | loss = 0.0000
Epoch 07 | weight = 4.9999 | prediction = 9.9999 | loss = 0.0000
Epoch 08 | weight = 5.0000 | prediction = 10.0000 | loss = 0.0000
Epoch 09 | weight = 5.0000 | prediction = 10.0000 | loss = 0.0000
Epoch 10 | weight = 5.0000 | prediction = 10.0000 | loss = 0.0000



You should see the learned weight move toward:

\[
w=5
\]

and the prediction move toward:

\[
10
\]

This is a tiny version of what happens when a neural network trains.



# 14. Day 2 Practical Exercise

Create:

```python
x = torch.tensor(3.0, requires_grad=True)
```

Then calculate:

\[
y = 2x^3 + 4x^2 + 3x + 1
\]

Use Autograd to determine:

\[
\frac{dy}{dx}
\]

at:

\[
x=3
\]

Try completing the next cell before looking at the solution.


In [16]:

# YOUR TURN

x = torch.tensor(3.0, requires_grad=True)

# Complete this:
y = 2*x**3 + 4*x**2 + 3*x + 1

# Calculate the gradient:
y.backward()

print("y =", y)
print("dy/dx =", x.grad)


y = tensor(100., grad_fn=<AddBackward0>)
dy/dx = tensor(81.)



### Manual solution

\[
y=2x^3+4x^2+3x+1
\]

Differentiate:

\[
\frac{dy}{dx}=6x^2+8x+3
\]

At:

\[
x=3
\]

\[
6(3^2)+8(3)+3
\]

\[
=54+24+3
\]

\[
=\boxed{81}
\]

Therefore PyTorch should give:

```text
tensor(81.)
```



# 15. Optional Challenge

Before running the code, calculate the gradients manually.

\[
f(x,y)=x^2y+3y^2
\]

Use:

\[
x=2,\quad y=3
\]

Find:

\[
\frac{\partial f}{\partial x}
\]

and:

\[
\frac{\partial f}{\partial y}
\]


In [17]:

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x**2 * y + 3 * y**2

f.backward()

print("f =", f)
print("df/dx =", x.grad)
print("df/dy =", y.grad)


f = tensor(39., grad_fn=<AddBackward0>)
df/dx = tensor(12.)
df/dy = tensor(22.)



## Challenge Solution

\[
f=x^2y+3y^2
\]

For \(x\):

\[
\frac{\partial f}{\partial x}=2xy
\]

At \(x=2,\ y=3\):

\[
2(2)(3)=12
\]

For \(y\):

\[
\frac{\partial f}{\partial y}=x^2+6y
\]

\[
=2^2+6(3)
\]

\[
=4+18=22
\]

Expected:

```text
df/dx = tensor(12.)
df/dy = tensor(22.)
```



# Day 2 Summary

The most important things to remember are:

### 1. Track gradients

```python
x = torch.tensor(3.0, requires_grad=True)
```

### 2. Build the computation normally

```python
y = x ** 2
```

### 3. Start backpropagation

```python
y.backward()
```

### 4. Read the gradient

```python
x.grad
```

### 5. Reset gradients when necessary

```python
x.grad.zero_()
```

### 6. Disable gradient tracking during inference or manual parameter updates

```python
with torch.no_grad():
    ...
```

### 7. Disconnect a tensor from the graph

```python
z = y.detach()
```

---

## Mental model

Think of Autograd as PyTorch doing this:

```text
inputs
  ↓
mathematical operations
  ↓
output / loss
  ↓
.backward()
  ↓
follow the graph backward
  ↓
calculate gradients
  ↓
use gradients to improve model parameters
```

## Next Step

After Autograd, the natural next topic is:

**Gradient Descent + `nn.Module`**

That is where we start turning these ideas into an actual PyTorch model.
